# 04 — RAG & Agent Metrics

**Needs `GROQ_API_KEY`.** (Free — see the folder README.)

---

## Real agents retrieve and act — so they need their own metrics

A plain Q&A metric (notebooks 01–02) treats the agent as a black box: question in,
answer out, grade the answer. But a **RAG agent** has moving parts:

```
question → [ retrieve documents ] → [ LLM reads them ] → answer
```

If the answer is wrong, *which part* failed? Bad retrieval? A grounded answer
built on the wrong documents? A hallucination ignoring good documents? You can't
tell from one score. So we measure each part.

### The RAG triad (the three questions that matter)
1. **Context relevance / retrieval quality** — did we fetch the *right* documents?
2. **Faithfulness (groundedness)** — is the answer *supported by* those documents,
   or did the model make things up?
3. **Answer relevance** — does the answer actually address the *question*?

Plus, for tool-using agents: **tool-call correctness** — did it call the right
tool with the right arguments?

We'll build all four over a tiny in-memory knowledge base (keyword retrieval — no
embeddings, no vector DB, so it runs instantly).

## Step 1 — Setup

In [ ]:
import os, re, json
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file (see the README)"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

def safe_parse(text: str) -> dict:
    """Pull the first JSON object out of an LLM response. Never raises."""
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return {"_raw": text, "_parse_error": True}

print("LLM ready:", llm.model_name)

## Step 2 — A tiny knowledge base + keyword retriever

Our 'corpus' is five short facts. The retriever scores each document by how many
query words it contains and returns the top-k. It's deliberately simple — and
deliberately *imperfect*, so our metrics have real mistakes to catch.

In [ ]:
KNOWLEDGE_BASE = [
    {"id": "d1", "text": "The Eiffel Tower is located in Paris and was completed in 1889."},
    {"id": "d2", "text": "The Great Wall of China is over 13,000 miles long."},
    {"id": "d3", "text": "Mount Everest is the tallest mountain on Earth at 8,849 meters."},
    {"id": "d4", "text": "The Amazon River is the largest river by discharge volume in the world."},
    {"id": "d5", "text": "Paris is the capital of France and home to the Louvre museum."},
]

def retrieve(query: str, k: int = 2) -> list[dict]:
    """Return the top-k docs by simple keyword overlap."""
    q_words = set(re.findall(r"[a-z]+", query.lower()))
    scored = []
    for doc in KNOWLEDGE_BASE:
        d_words = set(re.findall(r"[a-z]+", doc["text"].lower()))
        scored.append((len(q_words & d_words), doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:k]]

hits = retrieve("How tall is Mount Everest?")
for d in hits:
    print(f"  {d['id']}: {d['text']}")

## Step 3 — Metric 1: Retrieval quality (precision & recall @k)

Before grading any answer, ask: **did retrieval even fetch the right documents?**
If you have labeled 'relevant' doc ids for a question, this is pure arithmetic
(no LLM needed) — the same precision/recall idea from notebook 01, now over
*documents* instead of tokens:

- **Precision@k** = of the k docs we returned, how many were actually relevant?
- **Recall@k** = of all the relevant docs that exist, how many did we return?

In [ ]:
def retrieval_metrics(retrieved_ids: list[str], relevant_ids: list[str]) -> dict:
    retrieved, relevant = set(retrieved_ids), set(relevant_ids)
    hits = retrieved & relevant
    precision = len(hits) / len(retrieved) if retrieved else 0.0
    recall = len(hits) / len(relevant) if relevant else 0.0
    return {"precision": round(precision, 2), "recall": round(recall, 2)}

# 'How tall is Mount Everest?' — only d3 is truly relevant
retrieved = [d["id"] for d in retrieve("How tall is Mount Everest?")]
print("retrieved:", retrieved)
print("metrics  :", retrieval_metrics(retrieved, relevant_ids=["d3"]))

### Why this matters

`precision` < 1.0 means we pulled in junk documents that can distract the LLM.
`recall` < 1.0 means we **missed** a document the answer needed — no LLM can
recover a fact that was never retrieved. Most 'the RAG bot doesn't know X' bugs
are actually **recall** failures, and this metric is how you prove it.

## Step 4 — Build the RAG answer step

Now the generation half: feed the retrieved documents to the LLM and ask it to
answer **using only those documents**. We keep the retrieved context around — the
next two metrics grade the answer *against* it.

In [ ]:
RAG_PROMPT = """Answer the question using ONLY the context below. \
If the context doesn't contain the answer, say 'I don't know.'

CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""

def rag_answer(question: str, k: int = 2):
    docs = retrieve(question, k=k)
    context = "\n".join(f"- {d['text']}" for d in docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    answer = llm.invoke(prompt).content
    return answer, docs

answer, docs = rag_answer("Where is the Eiffel Tower and when was it built?")
print("ANSWER:", answer)

## Step 5 — Metric 2: Faithfulness (is the answer grounded?)

This is the metric that catches **hallucinations**. Faithfulness asks: *is every
claim in the answer supported by the retrieved context?* An answer can be fluent,
confident, and completely made up — faithfulness is your defense.

We use an LLM judge (from notebook 02) in a focused way: show it the context and
the answer, and ask it to rate how well the answer is supported. The judge does
**not** use outside knowledge — only the provided context counts as truth.

In [ ]:
FAITHFULNESS_PROMPT = """You are checking whether an ANSWER is fully supported by the CONTEXT.
Do NOT use outside knowledge. Only the CONTEXT counts as truth.
Rate how grounded the answer is, 1-5:
5 = every claim is directly supported by the context
3 = partly supported; some claims aren't in the context
1 = mostly unsupported / hallucinated

CONTEXT:
{context}

ANSWER: {answer}

Respond ONLY as JSON: {{"reasoning": "...", "faithfulness": <1-5>}}"""

def faithfulness_score(answer: str, docs: list[dict]) -> dict:
    context = "\n".join(d["text"] for d in docs)
    prompt = FAITHFULNESS_PROMPT.format(context=context, answer=answer)
    return safe_parse(llm.invoke(prompt).content)

# A grounded answer vs a hallucinated one, judged against the SAME context
grounded = "The Eiffel Tower is in Paris and was completed in 1889."
halluc  = "The Eiffel Tower is in Paris and was completed in 1925 by Gustave Eiffel's son."

print("grounded ->", faithfulness_score(grounded, docs).get("faithfulness"))
print("hallucinated ->", faithfulness_score(halluc, docs).get("faithfulness"))

## Step 6 — Metric 3: Answer relevance

An answer can be perfectly faithful to the context yet **not answer the question**
('The Eiffel Tower is in Paris' when you asked *when* it was built). Answer
relevance grades the answer against the **question**, independent of the context.

Faithfulness + answer relevance together close the loop: *grounded* AND *on-point*.

In [ ]:
ANSWER_RELEVANCE_PROMPT = """Rate how well the ANSWER addresses the QUESTION, 1-5:
5 = fully and directly answers the question
3 = partially answers, or answers a related but different question
1 = does not address the question at all

QUESTION: {question}
ANSWER: {answer}

Respond ONLY as JSON: {{"reasoning": "...", "relevance": <1-5>}}"""

def answer_relevance_score(question: str, answer: str) -> dict:
    prompt = ANSWER_RELEVANCE_PROMPT.format(question=question, answer=answer)
    return safe_parse(llm.invoke(prompt).content)

q = "When was the Eiffel Tower built?"
on_point = "It was completed in 1889."
off_topic = "The Eiffel Tower is located in Paris."
print("on-point  ->", answer_relevance_score(q, on_point).get("relevance"))
print("off-topic ->", answer_relevance_score(q, off_topic).get("relevance"))

## Step 7 — Metric 4: Tool-call correctness

Many agents don't just retrieve — they **call tools** (functions). To evaluate
that, you check the agent's chosen tool and arguments against what you *expected*.
This is programmatic (no LLM judge needed) and is the backbone of agent eval.

We simulate an agent that picks a tool for a query, then score its choice.

In [ ]:
AVAILABLE_TOOLS = ["search_web", "calculator", "get_weather"]

def tool_call_correctness(predicted: dict, expected: dict) -> dict:
    """Compare a predicted {tool, args} against the expected one."""
    tool_ok = predicted.get("tool") == expected.get("tool")
    # argument match: every expected arg present and equal (ignoring extras)
    exp_args = expected.get("args", {})
    pred_args = predicted.get("args", {})
    args_ok = all(pred_args.get(key) == val for key, val in exp_args.items())
    return {"tool_match": tool_ok, "args_match": args_ok, "correct": tool_ok and args_ok}

# Agent picked the calculator with the right expression — full marks
pred = {"tool": "calculator", "args": {"expression": "23*7"}}
exp  = {"tool": "calculator", "args": {"expression": "23*7"}}
print("correct call  ->", tool_call_correctness(pred, exp))

# Agent picked the wrong tool
pred_bad = {"tool": "search_web", "args": {"query": "23*7"}}
print("wrong tool    ->", tool_call_correctness(pred_bad, exp))

## Step 8 — A full RAG scorecard

Now we combine retrieval, faithfulness, and answer relevance into one **scorecard**
per question — the view you'd actually use to debug a RAG system. Each question
carries its labeled relevant doc ids so we can score retrieval too.

In [ ]:
rag_eval_set = [
    {"q": "How tall is Mount Everest?",                    "relevant": ["d3"]},
    {"q": "Where is the Eiffel Tower and when was it built?", "relevant": ["d1"]},
    {"q": "What is the capital of France?",                  "relevant": ["d5"]},
]

print(f"{'question':45} {'P':>4} {'R':>4} {'faith':>6} {'relev':>6}")
print('-' * 70)
scorecards = []
for case in rag_eval_set:
    answer, docs = rag_answer(case["q"])
    retrieved_ids = [d["id"] for d in docs]
    rm = retrieval_metrics(retrieved_ids, case["relevant"])
    faith = faithfulness_score(answer, docs).get("faithfulness", 0)
    relev = answer_relevance_score(case["q"], answer).get("relevance", 0)
    scorecards.append({**rm, "faithfulness": faith, "relevance": relev})
    print(f"{case['q'][:45]:45} {rm['precision']:>4} {rm['recall']:>4} {faith:>6} {relev:>6}")

## Step 9 — Aggregate the RAG scorecard

Average each dimension to get the system-level health check. Reading them
*together* localizes the problem: low recall → fix the retriever; high recall but
low faithfulness → tighten the prompt / model; high faithfulness but low relevance
→ the answer wanders off the question.

In [ ]:
n = len(scorecards)
agg = {
    "avg_precision":    round(sum(s['precision'] for s in scorecards) / n, 2),
    "avg_recall":       round(sum(s['recall'] for s in scorecards) / n, 2),
    "avg_faithfulness": round(sum(s['faithfulness'] for s in scorecards) / n, 2),
    "avg_relevance":    round(sum(s['relevance'] for s in scorecards) / n, 2),
}
print("RAG SYSTEM SCORECARD")
for k, v in agg.items():
    print(f"  {k:<18}: {v}")

## Step 10 — Try it yourself

1. **Break retrieval.** Set `k=1` in `rag_answer` and re-run Step 8 — watch recall
   drop on multi-fact questions, and faithfulness follow.
2. **Catch a hallucination live.** Ask a question the KB can't answer (e.g. 'Who
   built the Sphinx?') and confirm the agent says 'I don't know' (high faithfulness)
   rather than inventing an answer.
3. **Add a tool case.** Extend `tool_call_correctness` to score a list of expected
   tool calls in sequence (multi-step agents).

In [ ]:
# Your playground — test the 'I don't know' guardrail.
answer, docs = rag_answer("Who built the Great Sphinx of Giza?")
print("ANSWER:", answer)
print("faithfulness:", faithfulness_score(answer, docs).get("faithfulness"))

## Summary

| Metric | Question it answers | Needs LLM? |
|--------|--------------------|------------|
| `retrieval_metrics` (P/R@k) | Did we fetch the right documents? | No |
| `faithfulness_score` | Is the answer grounded in the context (no hallucination)? | Yes |
| `answer_relevance_score` | Does the answer address the question? | Yes |
| `tool_call_correctness` | Did the agent call the right tool with right args? | No |
| RAG scorecard | All of the above, per question + aggregated | Mixed |

**The big takeaway:** a single answer score hides *where* a RAG agent failed.
Split it into retrieval → faithfulness → relevance and each bad number points
straight at the component to fix.

**Next up → `05-eval-pipeline`:** we assemble every metric from notebooks 01–04
into one repeatable suite that A/B-tests two versions and **fails the build** when
quality regresses — the eval setup you'd actually run in CI.